# 05 · Hypothesis Testing — FRAUD_ECOMMERCE
**Marker:** `fraud_nb01-07_v2` · **Input:** `data/04_eda_complete.parquet` · **Output:** `data/05_hypothesis_done.parquet` (unchanged frame), `reports/05_feature_tests.csv`, `reports/05_run_record.json`

**Population: train rows only** (`__split == 'train'`). Nothing here fits a transform; results feed the feature contract in 06 and the leakage sentinels that 06 enforces.

**Run order:** 01 → 02 → 03 → 04 from this same package first. Notebook 02 (v2) computes the velocity columns point-in-time; sentinel S3 below verifies that, and notebook 06 refuses to run until it passes.

**Statistics at this scale.** With ~350k rows every non-trivial difference is "significant". Features are ranked by **effect size** (|rank-biserial| for numeric, Cramér's V for categorical); p-values are Benjamini–Hochberg corrected and reported, not used to select.

In [ ]:
%pip install -q boto3==1.43.95

In [ ]:
# MARKER: fraud_nb01-07_v2 :: 05_Hypothesis_Testing
import io, os, json, time, hashlib, platform, importlib
from datetime import datetime, timezone
import boto3
from botocore.exceptions import ClientError
import joblib
import numpy as np
import pandas as pd
from google.colab import userdata

BUCKET, REGION = "fraud-ecommerce", "ap-south-2"
SPLIT_DATE = pd.Timestamp("2025-07-01")      # stamped in notebook 01 as __split; verified here, never recomputed
CONTRACT_VERSION = "v1"
SEED = 42
MARKER = "fraud_nb01-07_v2"
for _k in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY"):
    if not os.environ.get(_k):
        os.environ[_k] = userdata.get(_k)     # Colab Secrets -> process env only; never printed or saved
s3 = boto3.client("s3", region_name=REGION)
RAW, LABELS, CONTRACTS, DATA, REPORTS = "raw/", "raw/label_sources/", "contracts/", "data/", "reports/"  # flat layout
ident = boto3.client("sts", region_name=REGION).get_caller_identity()
print("account:", ident["Account"], "| arn:", ident["Arn"])
if ident["Arn"].endswith(":root"):
    print("NOTE: running as root — accepted for Phase 1; move to an IAM principal before Phase 2")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)


def _jsonable(o):
    if isinstance(o, (np.integer, np.floating, np.bool_)):
        return o.item()
    if isinstance(o, (pd.Timestamp, datetime)):
        return o.isoformat()
    if isinstance(o, np.ndarray):
        return o.tolist()
    raise TypeError(f"not JSON-serialisable: {type(o).__name__}")


def read_bytes_s3(key):
    return s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()


def put_bytes_s3(body, key):
    s3.put_object(Bucket=BUCKET, Key=key, Body=body)
    print(f"saved s3://{BUCKET}/{key}  ({len(body):,} bytes)")


def key_exists(key):
    try:
        s3.head_object(Bucket=BUCKET, Key=key)
        return True
    except ClientError as e:
        if e.response["Error"]["Code"] in ("404", "NoSuchKey", "NotFound"):
            return False
        raise


def read_s3(key):
    return pd.read_parquet(io.BytesIO(read_bytes_s3(key)))


def save_s3(df, key):
    """Parquet only; the bytes are verified to round-trip columns, dtypes and categories before upload."""
    buf = io.BytesIO()
    df.to_parquet(buf, index=False)
    body = buf.getvalue()
    back = pd.read_parquet(io.BytesIO(body))
    assert list(back.columns) == list(df.columns) and len(back) == len(df), f"parquet round-trip changed shape: {key}"
    bad = [c for c in df.columns if str(back[c].dtype) != str(df[c].dtype)]
    assert not bad, f"parquet round-trip changed dtypes in {key}: {bad}"
    badcat = [c for c in df.columns if str(df[c].dtype) == "category"
              and list(back[c].cat.categories) != list(df[c].cat.categories)]
    assert not badcat, f"parquet round-trip changed categories in {key}: {badcat}"
    put_bytes_s3(body, key)


def read_json_s3(key):
    return json.loads(read_bytes_s3(key))


def save_json_s3(obj, key):
    put_bytes_s3(json.dumps(obj, indent=1, default=_jsonable).encode(), key)


def save_model_s3(obj, key):
    buf = io.BytesIO()
    joblib.dump(obj, buf)
    put_bytes_s3(buf.getvalue(), key)


def load_model_s3(key):
    return joblib.load(io.BytesIO(read_bytes_s3(key)))


# names used by notebooks 01-04 (same verified implementations underneath)
def s3_read_csv(key, **kw):
    return pd.read_csv(io.BytesIO(read_bytes_s3(key)), **kw)


s3_read_parquet, s3_read_json = read_s3, read_json_s3


def s3_write_parquet(df, key):
    save_s3(df, key)
    return f"s3://{BUCKET}/{key}"


def s3_write_json(obj, key):
    save_json_s3(obj, key)
    return f"s3://{BUCKET}/{key}"


RAW_KEYS = ([f"{RAW}{t}.csv" for t in ["payments", "orders", "order_items", "account_logins", "customers",
                                         "merchants", "cards", "devices", "ip_reputation"]]
            + [f"{LABELS}{t}.csv" for t in ["fraud_events", "audit_sample", "chargebacks"]]
            + [f"{CONTRACTS}schema_v1.json"])
_absent = [k for k in RAW_KEYS if not key_exists(k)]
assert not _absent, f"missing landing objects in s3://{BUCKET}/: {_absent}"
print(f"landing objects present: {len(RAW_KEYS)} (flat layout, bucket root)")


def run_meta(notebook):
    return {"notebook": notebook, "marker": MARKER,
            "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "library_versions": LIB_VERSIONS, "version_drift": VERSION_DRIFT}


LIBS = ["pandas", "numpy", "pyarrow", "scipy", "statsmodels", "sklearn", "lightgbm", "xgboost", "joblib", "boto3"]
LIB_VERSIONS = {"python": platform.python_version(),
                **{m: importlib.import_module(m).__version__ for m in LIBS}}
EXPECTED = {"pandas": "2.2.3", "numpy": "2.1.3", "pyarrow": "23.0.1", "scipy": "1.16.3", "statsmodels": "0.15.0",
            "sklearn": "1.6.1", "lightgbm": "4.6.0", "xgboost": "3.4.1", "boto3": "1.43.95"}
VERSION_DRIFT = {m: {"verified": v, "found": LIB_VERSIONS[m]} for m, v in EXPECTED.items() if LIB_VERSIONS[m] != v}
if not LIB_VERSIONS["python"].startswith("3.13."):
    VERSION_DRIFT["python"] = {"verified": "3.13.x", "found": LIB_VERSIONS["python"]}
print(LIB_VERSIONS)
print("VERSION DRIFT vs the runtime verified on 2026-09-16:", VERSION_DRIFT or "none")

In [ ]:
# 05.1 Load stage 04 and verify the invariants this notebook relies on (nothing is recomputed)
df = read_s3("data/04_eda_complete.parquet")
print(df.shape)
assert df.shape[1] == 110, f"expected 110 columns from stage 04, found {df.shape[1]}"
assert set(df["__split"].unique()) == {"train", "test"}, df["__split"].unique()
_tr_max = df.loc[df["__split"].eq("train"), "payment_ts"].max()
_te_min = df.loc[df["__split"].eq("test"), "payment_ts"].min()
assert _tr_max < SPLIT_DATE <= _te_min, f"stamped __split disagrees with {SPLIT_DATE.date()}: {_tr_max} / {_te_min}"

# locked label definition: (forced is_fraud value or None, weight)
LABEL_RULES = {"alert_review": (None, 1.0), "random_audit": (None, 1.0),
               "chargeback": (1, 0.6), "unadjudicated": (0, 0.35)}
assert set(df["label_source"].unique()) == set(LABEL_RULES), df["label_source"].unique()
for src, (lab, w) in LABEL_RULES.items():
    m = df["label_source"].eq(src)
    assert df.loc[m, "sample_weight"].eq(w).all(), f"{src}: sample_weight != {w}"
    if lab is not None:
        assert df.loc[m, "is_fraud"].eq(lab).all(), f"{src}: is_fraud != {lab}"

baseline_score = pd.to_numeric(df["payment_risk_score"].str.strip(), errors="raise").astype("float64")
alerted_int = df["alerted_flag"].map({"Y": 1, "N": 0})
assert alerted_int.notna().all(), df["alerted_flag"].unique()

tr = df["__split"].eq("train")
T = df.loc[tr].copy()
T["baseline_score"] = baseline_score[tr]
T["alerted_int"] = alerted_int[tr].astype("int64")
y = T["is_fraud"].to_numpy()
pos = y == 1
print(f"train rows {len(T):,} | positives {int(pos.sum()):,} | observed rate {y.mean():.4%}")

In [ ]:
TREE_NUMERIC = [
    "payment_amount", "processing_fee", "discount_amount", "item_count", "attempt_seq_in_session",
    "is_guest_checkout", "is_3ds_attempted", "is_3ds_success",
    "address_match_flag", "shipping_addr_age_hours", "kyc_level", "city_tier",
    "prior_return_rate", "prior_orders_12m", "avg_ticket_size", "trailing_chargeback_rate_bps",
    "is_emulator", "reputation_score", "n_categories", "max_unit_price", "total_qty",
    "last_login_new_device", "last_login_unusual_loc", "last_login_failed_attempts", "last_login_risk",
    "hours_since_last_login", "card_token_age_h", "device_age_h", "account_age_days",
    "amount_vs_merchant_ticket", "ip_country_mismatch", "ip_country_missing", "ip_region_mismatch",
    "issuer_foreign", "has_coupon", "hour_of_day", "day_of_week",
    "device_n_customers", "ip_n_customers",
    "has_card", "has_device_profile", "has_basket", "has_prior_login",
]
TREE_CATEGORICAL = [
    "payment_method", "payment_gateway", "shipping_speed", "delivery_type", "email_domain_class",
    "acquisition_channel", "merchant_category", "network", "issuer", "product_type",
    "device_type", "browser_family", "asn_type", "asn_country",
]


# 05 tests the model candidates plus three columns 06 will reduce to flags (reported for completeness)
NUMERIC = TREE_NUMERIC
CATEGORICAL = TREE_CATEGORICAL + ["ip_country", "issuing_country", "city"]
_missing = [c for c in NUMERIC + CATEGORICAL if c not in df.columns]
assert not _missing, f"candidate columns missing from stage 04: {_missing}"
print(len(NUMERIC), "numeric |", len(CATEGORICAL), "categorical candidates")

In [ ]:
def asof_distinct_count(frame, key, entity="customer_id", ts="payment_ts"):
    """Distinct `entity` values seen on `key` at or before each row's `ts` (inclusive). NaN where `key` is null.
    Point-in-time by construction: a row never sees a later first appearance, so no split boundary is needed."""
    f = frame.loc[frame[key].notna(), [key, entity, ts]]
    first = (f.groupby([key, entity], as_index=False, sort=True)[ts].min()
               .sort_values(ts, kind="mergesort"))
    first["_n"] = first.groupby(key, sort=False).cumcount() + 1
    rows = f.assign(_row=f.index).sort_values(ts, kind="mergesort")
    m = pd.merge_asof(rows, first[[key, ts, "_n"]], on=ts, by=key,
                      direction="backward", allow_exact_matches=True)
    assert m["_n"].notna().all(), f"as-of join left gaps for {key}"
    return m.set_index("_row")["_n"].reindex(frame.index).astype("float64")


VELOCITY = {"device_n_customers": "device_id", "ip_n_customers": "ip_address"}


# 05.2 Sentinel S3 input — are the velocity columns point-in-time? (computed on all rows; no fitting involved)
velocity_check = {}
for col, key in VELOCITY.items():
    pit = asof_distinct_count(df, key)
    eq = df[col].astype("float64").fillna(-1).eq(pit.fillna(-1))
    velocity_check[col] = {"rows_equal_share": round(float(eq.mean()), 6), "is_point_in_time": bool(eq.all())}
print(velocity_check)
if not all(v["is_point_in_time"] for v in velocity_check.values()):
    print("S3 FLAG: velocity columns are NOT point-in-time -> stage 04 was not produced by notebook 02 v2 (06 will refuse to run)")

In [ ]:
# 05.3 Univariate association with the hard union label (train rows, unweighted)
from scipy.stats import mannwhitneyu, chi2_contingency, false_discovery_control

rows = []
for c in NUMERIC:
    x = T[c].astype("float64").to_numpy()
    ok = ~np.isnan(x)
    a, b = x[ok & pos], x[ok & ~pos]
    u = mannwhitneyu(a, b, alternative="two-sided", method="asymptotic")
    auc = u.statistic / (len(a) * len(b))              # P(fraud value > legit value)
    rows.append({"feature": c, "kind": "numeric", "n": int(ok.sum()), "p": float(u.pvalue),
                 "effect_name": "|rank-biserial|", "effect": abs(2 * auc - 1), "auc_fraud_higher": auc,
                 "median_fraud": float(np.median(a)), "median_legit": float(np.median(b)),
                 "missing_fraud": float(np.isnan(x[pos]).mean()), "missing_legit": float(np.isnan(x[~pos]).mean())})
for c in CATEGORICAL:
    s = T[c].astype(object).where(T[c].notna(), "__MISSING__")
    ct = pd.crosstab(s, y)
    chi2, p, dof, _ = chi2_contingency(ct)
    n = int(ct.to_numpy().sum())
    rate = ct[1] / ct.sum(axis=1)
    top = rate.idxmax()
    rows.append({"feature": c, "kind": "categorical", "n": n, "p": float(p),
                 "effect_name": "Cramer's V", "effect": float(np.sqrt(chi2 / (n * (min(ct.shape) - 1)))),
                 "levels": int(ct.shape[0]), "top_level": str(top),
                 "top_level_rate": float(rate[top]), "top_level_n": int(ct.sum(axis=1)[top])})
tests = pd.DataFrame(rows)
tests["p_bh"] = false_discovery_control(tests["p"].to_numpy(), method="bh")
tests = tests.sort_values("effect", ascending=False).reset_index(drop=True)
print(tests[["feature", "kind", "effect_name", "effect", "p_bh", "auc_fraud_higher",
             "median_fraud", "median_legit", "top_level", "top_level_rate"]].round(4).to_string())
print(f"\n{int((tests.p_bh < 0.05).sum())} of {len(tests)} significant after BH at 0.05 — expected at n={len(T):,}; "
      "rank by effect (rank-biserial: ~0.1 small, ~0.3 medium, ~0.5 large).")
put_bytes_s3(tests.to_csv(index=False).encode(), "reports/05_feature_tests.csv")

In [ ]:
# 05.4 Pre-registered hypotheses (6 tests -> Bonferroni alpha = 0.05 / 6)
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

ALPHA = 0.05 / 6
H = {}


def two_group(name, mask, labels, label_on, label_off, h0):
    g = pd.DataFrame({"on": mask.to_numpy(), "y": labels}).groupby("on")["y"].agg(["sum", "size"])
    assert set(g.index) == {False, True}, f"{name}: one group is empty"
    z, p = proportions_ztest(g["sum"].to_numpy(), g["size"].to_numpy())
    r_on, r_off = g.loc[True, "sum"] / g.loc[True, "size"], g.loc[False, "sum"] / g.loc[False, "size"]
    H[name] = {"h0": h0, "test": "two-proportion z", "rate_" + label_on: r_on, "rate_" + label_off: r_off,
               "n_" + label_on: int(g.loc[True, "size"]), "lift": r_on / r_off if r_off > 0 else None,
               "z": float(z), "p": float(p), "reject_h0": bool(p < ALPHA)}
    print(f"{name}: {label_on} {r_on:.3%} vs {label_off} {r_off:.3%} (lift {H[name]['lift']:.2f}), "
          f"p={p:.2e} -> {'reject' if p < ALPHA else 'retain'} H0: {h0}")


# H1 fraud rate is independent of payment method
ct = pd.crosstab(T["payment_method"], T["is_fraud"])
chi2, p, dof, _ = chi2_contingency(ct)
rates = (ct[1] / ct.sum(axis=1)).sort_values(ascending=False)
H["H1_payment_method"] = {"h0": "fraud rate independent of payment_method", "test": "chi-square",
                          "chi2": float(chi2), "dof": int(dof), "p": float(p), "reject_h0": bool(p < ALPHA),
                          "rates": rates.round(5).to_dict()}
print(f"H1_payment_method: chi2={chi2:.1f} dof={dof} p={p:.2e} -> {'reject' if p < ALPHA else 'retain'} H0")
print(rates.round(5).to_string())

# H2 among card payments, a successful 3DS challenge does not change fraud rate
card = T["has_card"].eq(1)
two_group("H2_3ds_success_cards", T.loc[card, "is_3ds_success"].eq(1), y[card.to_numpy()], "3ds_ok", "no_3ds_ok",
          "3DS success does not change fraud rate on card payments")

# H3 account age distribution is the same for fraud and legitimate payments
a = T.loc[pos, "account_age_days"]; b = T.loc[~pos, "account_age_days"]
u = mannwhitneyu(a, b, alternative="two-sided", method="asymptotic")
H["H3_account_age"] = {"h0": "account_age_days identically distributed across classes", "test": "Mann-Whitney U",
                       "median_fraud": float(a.median()), "median_legit": float(b.median()),
                       "rank_biserial": float(2 * u.statistic / (len(a) * len(b)) - 1),
                       "p": float(u.pvalue), "reject_h0": bool(u.pvalue < ALPHA)}
print(f"H3_account_age: median fraud {a.median():.1f}d vs legit {b.median():.1f}d, "
      f"r={H['H3_account_age']['rank_biserial']:.3f}, p={u.pvalue:.2e} -> "
      f"{'reject' if u.pvalue < ALPHA else 'retain'} H0")

two_group("H4_ip_country_mismatch", T["ip_country_mismatch"].eq(1), y, "mismatch", "match",
          "IP-country mismatch does not change fraud rate")
two_group("H5_shared_device", T["device_n_customers"].gt(1), y, "shared", "single",
          "a device already used by another customer does not change fraud rate")
two_group("H6_guest_checkout", T["is_guest_checkout"].eq(1), y, "guest", "account",
          "guest checkout does not change fraud rate")

In [ ]:
# 05.5 Leakage sentinels (recorded; notebook 06 refuses to run while any is flagged and unacknowledged)
from sklearn.metrics import average_precision_score

SENTINELS = {}
_num = tests[tests.kind.eq("numeric")]
s1_num = _num.loc[np.maximum(_num.auc_fraud_higher, 1 - _num.auc_fraud_higher) >= 0.95, "feature"].tolist()
_cat = tests[tests.kind.eq("categorical")]
s1_cat = _cat.loc[(_cat.top_level_rate >= 0.95) & (_cat.top_level_n >= 50), "feature"].tolist()
SENTINELS["S1_near_perfect_separation"] = {
    "flag": bool(s1_num or s1_cat), "numeric": s1_num, "categorical": s1_cat,
    "rule": "univariate AUC >= 0.95, or a level with n >= 50 and fraud rate >= 95%"}

# S2 processing_fee is contract-flagged pre-decision, but a zero fee could encode a failed/blocked payment
nc = T["payment_method"].ne("Cash on Delivery").to_numpy()
zero = T["processing_fee"].eq(0).to_numpy()
tab = pd.crosstab(zero[nc], y[nc])
if True in tab.index and False in tab.index:
    chi2, p_fee, _, _ = chi2_contingency(tab)
    r_zero = tab.loc[True, 1] / tab.loc[True].sum()
    r_fee = tab.loc[False, 1] / tab.loc[False].sum()
    ratio = r_zero / r_fee
    flag = bool((ratio > 3 or ratio < 1 / 3) and p_fee < 1e-3)
    SENTINELS["S2_zero_fee_non_cod"] = {"flag": flag, "zero_fee_rows": int(tab.loc[True].sum()),
                                        "rate_zero_fee": float(r_zero), "rate_with_fee": float(r_fee),
                                        "ratio": float(ratio), "p": float(p_fee),
                                        "rule": "ratio outside [1/3, 3] and p < 1e-3"}
else:
    SENTINELS["S2_zero_fee_non_cod"] = {"flag": False, "note": "no zero-fee (or no fee) non-COD rows in train"}

SENTINELS["S3_velocity_not_point_in_time"] = {
    "flag": not all(v["is_point_in_time"] for v in velocity_check.values()), **velocity_check,
    "fix": "re-run notebooks 02 -> 03 -> 04 -> 05 from package fraud_nb01-07_v2"}

for k, v in SENTINELS.items():
    print(("FLAG  " if v["flag"] else "ok    ") + k, {kk: vv for kk, vv in v.items() if kk != "flag"})

# Context for the gate design: which population does each label source actually describe?

def design_weights(label_source, alerted, in_scope):
    """Inverse-probability weights for the two-stratum review design inside `in_scope` (numpy bool array).
    Alerted stratum: reviewed through alert_review (near-census). Non-alerted stratum: random_audit sample.
    Rows outside the design (chargeback-only, unadjudicated) get weight 0."""
    label_source, alerted = np.asarray(label_source), np.asarray(alerted)
    assert not ((label_source == "random_audit") & (alerted == 1)).any(), "audit rows must be non-alerted"
    assert not ((label_source == "alert_review") & (alerted == 0)).any(), "alert-review rows must be alerted"
    w = np.zeros(len(label_source), dtype="float64")
    for flag, src in ((1, "alert_review"), (0, "random_audit")):
        stratum = in_scope & (alerted == flag)
        sampled = stratum & (label_source == src)
        assert sampled.sum() > 0, f"no {src} rows in scope"
        w[sampled] = stratum.sum() / sampled.sum()
    return w


def ap_ipw(y, score, w):
    """Design-weighted average precision: an estimate of population PR-AUC from the reviewed sample."""
    s = w > 0
    return float(average_precision_score(y[s], score[s], sample_weight=w[s]))

aud = T["label_source"].eq("random_audit").to_numpy()
src_t, alert_t = T["label_source"].to_numpy(), T["alerted_int"].to_numpy()
DESIGN = {"audit_rows_alerted": int((aud & (alert_t == 1)).sum()),
          "alert_review_rows_not_alerted": int(((src_t == "alert_review") & (alert_t == 0)).sum()),
          "alerted_rows_not_reviewed": int(((src_t != "alert_review") & (alert_t == 1)).sum())}
w_design = design_weights(src_t, alert_t, np.ones(len(T), dtype=bool))
DESIGN["ipw_prevalence_train"] = float((w_design * y).sum() / w_design.sum())
DESIGN["incumbent_ap_ipw_train"] = ap_ipw(y, T["baseline_score"].to_numpy(), w_design)
print("review design (train):", DESIGN)
print("random_audit samples the NON-alerted stratum only; alert_review covers the alerted stratum. The unbiased population "
      "estimate weights each stratum by its inverse sampling fraction (07 gates on this).")
k_aud, n_aud = int(y[aud].sum()), int(aud.sum())
lo, hi = proportion_confint(k_aud, n_aud, alpha=0.05, method="wilson")
INCUMBENT = {
    "ap_all_train": float(average_precision_score(y, T["baseline_score"])),
    "prevalence_all_train": float(y.mean()),
    "ap_audit_train": float(average_precision_score(y[aud], T.loc[aud, "baseline_score"])),
    "audit_rows_train": n_aud, "audit_positives_train": k_aud,
    "audit_prevalence_train": k_aud / n_aud, "audit_prevalence_wilson95": [lo, hi]}
INCUMBENT["lift_all"] = INCUMBENT["ap_all_train"] / INCUMBENT["prevalence_all_train"]
INCUMBENT["lift_audit"] = INCUMBENT["ap_audit_train"] / INCUMBENT["audit_prevalence_train"]
print("\nincumbent payment_risk_score on train:", {k: (round(v, 5) if isinstance(v, float) else v)
                                                   for k, v in INCUMBENT.items()})
print("Lift over prevalence is the comparable quantity (AP depends on prevalence). All-row lift is inflated: "
      "alert_review labels exist because the incumbent fired. Audit-row lift covers only the non-alerted stratum.")

In [ ]:
# 05.6 Temporal stability inside train (early = 2024, late = 2025-H1) — PSI; test rows are not used
EARLY = (T["payment_ts"] < pd.Timestamp("2025-01-01")).to_numpy()


def psi_from_shares(r, c):
    r = np.clip(r, 1e-6, None); c = np.clip(c, 1e-6, None)
    return float(np.sum((c - r) * np.log(c / r)))


def psi_categorical(ref, cur):
    levels = sorted(set(ref.unique()) | set(cur.unique()), key=str)
    r = ref.value_counts(normalize=True).reindex(levels, fill_value=0).to_numpy()
    c = cur.value_counts(normalize=True).reindex(levels, fill_value=0).to_numpy()
    return psi_from_shares(r, c)


def psi_numeric(ref, cur, bins=10):
    ref, cur = ref[~np.isnan(ref)], cur[~np.isnan(cur)]
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    r = np.histogram(ref, edges)[0] / len(ref)
    c = np.histogram(cur, edges)[0] / len(cur)
    return psi_from_shares(r, c)


psi_rows = []
for c in NUMERIC:
    x = T[c].astype("float64")
    if x.nunique() <= 10:
        v = psi_categorical(x[EARLY].astype(str), x[~EARLY].astype(str))
    else:
        v = psi_numeric(x[EARLY].to_numpy(), x[~EARLY].to_numpy())
    psi_rows.append({"feature": c, "psi": v, "missing_early": float(x[EARLY].isna().mean()),
                     "missing_late": float(x[~EARLY].isna().mean())})
for c in CATEGORICAL:
    s = T[c].astype(object).where(T[c].notna(), "__MISSING__").astype(str)
    psi_rows.append({"feature": c, "psi": psi_categorical(s[EARLY], s[~EARLY]), "missing_early": None, "missing_late": None})
psi = pd.DataFrame(psi_rows).sort_values("psi", ascending=False)
psi["band"] = pd.cut(psi["psi"], [-np.inf, 0.1, 0.25, np.inf], labels=["stable", "moderate", "significant"])
print(psi.round(4).to_string(index=False))
print("\nCalendar-driven features (ages) drift by construction as the population ages; trees extrapolate flat "
      "beyond the training maximum. This is a monitoring item, not a leakage finding.")

In [ ]:
# 05.7 Save the unchanged frame (keeps the chain intact) and the run record
assert df.shape[1] == 110 and "baseline_score" not in df.columns
save_s3(df, "data/05_hypothesis_done.parquet")

record = {
    "notebook": "05_Hypothesis_Testing", "marker": MARKER,
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "population": "train rows only", "rows": int(len(df)), "cols": int(df.shape[1]),
    "train_rows": int(len(T)), "train_positives": int(pos.sum()),
    "tests": {"n_features": int(len(tests)), "n_bh_significant_0.05": int((tests.p_bh < 0.05).sum()),
              "selection_rule": "none — effect sizes are reported; the feature contract is fixed by name in 06",
              "top_by_effect": tests.head(15)[["feature", "kind", "effect_name", "effect", "p_bh"]].to_dict("records")},
    "hypotheses": H, "bonferroni_alpha": ALPHA,
    "sentinels": SENTINELS,
    "incumbent_train": INCUMBENT, "review_design_train": DESIGN,
    "temporal_psi": psi[psi["band"] != "stable"][["feature", "psi"]].to_dict("records"),
    "library_versions": LIB_VERSIONS, "version_drift": VERSION_DRIFT, "columns_added": 0,
}
save_json_s3(record, "reports/05_run_record.json")
print("\nSENTINELS:", {k: v["flag"] for k, v in SENTINELS.items()})